# Cross validation
Demonstration of the benefit of using the sklearn framework

## 1. Import Packages

Below, we import both standard packages, and functions from the accompanying .py files

In [ ]:
# Import standard packages

import os
import pickle
import sys

import numpy as np
from scipy import io, stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetRegressor
from torch import optim
from tqdm import tqdm
import spikeinterface.full as si

from Neural_Decoding.nn import FNN
from Neural_Decoding.preprocessing_funcs import LagMat, bin_spikes, bin_output

import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from collections import defaultdict


## 2. Load Data
This notebook now formats multi-unit activity (MUA) from `peaks_combined2_sorted.npy` by treating each channel as one neuron.

The neural input you will build is a matrix of size `number of time bins` x `number of channels`, where each entry is the number of spikes from that channel in that bin.

If you want to run decoding, load your own behavioral target separately so that `y` matches the same time bins as `neural_data`.

In [33]:
###Load Recording###
recording_path = r"/gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/preprocessed"
rec = si.load(recording_path)
print(rec)

###Load Data###
peaks_path = r"/gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/preprocessed/thresholding_peaks_3.5/peaks_combined2_sorted.npy" 
peaks = np.load(peaks_path, allow_pickle=True) #Load peaks of all neurons
print(peaks.dtype.names)

BinaryFolderRecording: 355 channels - 30000.283521 Hz - 1 segments - 80,418,082 samples 
                       2,680.58s (44.68 minutes) - float16 dtype - 53.18 GiB
('sample_index', 'channel_index', 'amplitude', 'segment_index', 'z', 'matched_filter_score')


#### 2a. Format peaks into channel-wise MUA
Use one neuron per channel: spikes from the same channel are grouped together, converted to seconds, and binned into `neural_data`.

In [34]:
channel_ids = rec.get_channel_ids()
sampling_frequency = rec.get_sampling_frequency()
print(channel_ids.shape)
print(f"sampling_frequency = {sampling_frequency}")

(355,)
sampling_frequency = 30000.283520599252


In [35]:
peak_channel_index = peaks['channel_index'].astype(int)
peak_times_sec = peaks['sample_index'].astype(float) / sampling_frequency

# Treat each channel as one neuron (MUA): group peak times by channel
n_channels = channel_ids.shape[0]
if peak_channel_index.min() >= 0 and peak_channel_index.max() < n_channels:
    peak_row_index = peak_channel_index
    mapping_mode = "direct row index"
else:
    # Fallback: map absolute AP channel numbers (e.g. AP354) to row indices
    ap_numbers = np.array([int(str(cid).split('AP')[-1]) for cid in channel_ids], dtype=int)
    ap_to_row = {ap: i for i, ap in enumerate(ap_numbers)}
    missing = np.setdiff1d(np.unique(peak_channel_index), ap_numbers)
    if missing.size > 0:
        raise ValueError(
            f"Cannot map {missing.size} channel_index values to channel_ids. "
            f"First missing values: {missing[:10]}"
        )
    peak_row_index = np.array([ap_to_row[ch] for ch in peak_channel_index], dtype=int)
    mapping_mode = "mapped AP number -> row index"

spike_times = np.asarray(
    [peak_times_sec[peak_row_index == ch] for ch in range(n_channels)],
    dtype=object,
 )

dt = 0.2  # bin size in seconds
t_start = float(rec.get_times()[0])
t_end = float(rec.get_times()[-1])

neural_data = bin_spikes(spike_times, dt, t_start, t_end)

print(f"Mapping mode: {mapping_mode}")
print(f"n_channels = {n_channels}")
print(f"spike_times is an object array of length: {spike_times.shape[0]}")
print(f"neural_data shape (time bins x channels): {neural_data.shape}")
print(f"t_start = {t_start}, t_end = {t_end}, dt = {dt}")
print(f"Total spikes binned: {int(neural_data.sum())}")

Mapping mode: direct row index
n_channels = 355
spike_times is an object array of length: 355
neural_data shape (time bins x channels): (13402, 355)
t_start = 7.414796591747566, t_end = 2687.992163295036, dt = 0.2
Total spikes binned: 1890417


## 3. Preprocess Data

### 3A. User Inputs
The user can define what time period to use spikes from (with respect to the output).

In [3]:
BIN_BEFORE = 6  # How many bins of neural data prior to the output are used for decoding
BIN_CURRENT = 1  # Whether to use concurrent time bin of neural data
BIN_AFTER = 6  # How many bins of neural data after the output are used for decoding

### 3B. Format Covariates

In [ ]:
# Set decoding input
X = neural_data.astype("float32")  # torch requires float32

# Set decoding output here for your own decoding task
# y = ...  # must have the same number of time bins as neural_data

## 4. Run cross validation

#### User Options

KFold cross-validation

In [22]:
N_SPLITS = 3

In [27]:
models = {
    "WF": {
        "estimator": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("lagmat", LagMat(BIN_BEFORE, BIN_CURRENT, BIN_AFTER, flat=True)),
                ("linear", LinearRegression()),
            ]
        ),
    },
    "FNN": {
        "estimator": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("lagmat", LagMat(BIN_BEFORE, BIN_CURRENT, BIN_AFTER, flat=True)),
                (
                    "fnn",
                    NeuralNetRegressor(
                        module=FNN,
                        lr=0.001,
                        iterator_train__shuffle=True,
                        optimizer=optim.Adam,
                        batch_size=32,
                        module__n_targets=y.shape[1],
                        module__num_units=400,
                        module__frac_dropout=0.25,
                        module__n_layers=2,
                        max_epochs=10,
                        verbose=0,
                    ),
                ),
            ]
        ),
    },
}

for name in tqdm(models):
    model = models[name]["estimator"]

    cv_results = cross_validate(
        model,
        X,
        y,
        cv=KFold(n_splits=N_SPLITS),
        scoring="r2",
        n_jobs=1,  # parallelisation
        verbose=1,
    )

    print(f"R2s {name}: {cv_results['test_score']}")

 50%|██████████████████████████████████████████▌                                          | 1/2 [00:01<00:01,  1.57s/it]

R2s WF: [0.73822659 0.7493962  0.71846437]


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:   42.8s finished
100%|█████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:44<00:00, 22.19s/it]

R2s FNN: [0.83662546 0.84977973 0.83463073]
